In [60]:
%use lets-plot

import application.tester.TradingAlgorithmBackTester
import data.repository.historical_data.HistoricalMarketDataProvider
import domain.algorithm.TradingAlgorithm
import domain.market.security.SecurityIdentifier
import domain.tax.Taxation
import domain.trader.TradingOrder
import kotlinx.coroutines.async
import kotlinx.coroutines.awaitAll
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.runBlocking
import kotlin.time.Instant

enum class TradingOrderColor {
    BUY,
    SELL,
    BOTH,
    NOTHING,
}

//===========================================================//
//===========================================================//

// Config
val algorithm = TradingAlgorithm.Type.TACPP46
val taxation = Taxation.Type.Hungary
val startCapital = 1000.0
val startDate = Instant.parse("2025-06-01T00:00:00Z")
val endDate = Instant.parse("2026-01-01T00:00:00Z")

val listOfOutput = runBlocking {
    coroutineScope {
        HistoricalMarketDataProvider.getAllSecurityIdentifiers().getOrThrow().map {
            async {
                TradingAlgorithmBackTester(
                    type = algorithm,
                    securityIdentifier = it,
                    startingCapital = startCapital,
                    taxation = taxation,
                    from = startDate,
                    to = endDate
                ).runBackTest()
            }
        }.awaitAll()
    }
}

val plots = listOfOutput.map { output ->
    val days = output.stockHistory.mapIndexed { index, _ -> index }
    val stockPrice = output.stockHistory.map { it.closingPrice }
    val tradingOrders = output.tradingOrders.map { order ->
        if(order.buy == null && order.sell == null) TradingOrderColor.NOTHING
        else if(order.buy != null && order.sell != null) TradingOrderColor.BOTH
        else if(order.buy != null && order.sell == null) TradingOrderColor.BUY
        else TradingOrderColor.SELL
    }

    val plotData = mapOf(
        "day" to days,
        "stock_price" to stockPrice,
        "trading_orders" to tradingOrders
    )

    val pointIndices = tradingOrders.indices .filter { tradingOrders[it] != TradingOrderColor.NOTHING }
    val pointData = mapOf(
        "day" to pointIndices.map { days[it] },
        "stock_price" to pointIndices.map { stockPrice[it] },
        "trading_orders" to pointIndices.map { tradingOrders[it].name }
    )

    letsPlot(plotData) { x = "day"; y = "stock_price" } +
    geomLine(color = "steelblue", size = 1.2) +
    geomPoint(
        data = pointData,
        size = 2.0
    ) {
        //color = "trading_orders" WTF IS WRONG HEEEEEREEEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE
    } +
    scaleColorManual(
        values = mapOf(
            TradingOrderColor.BUY.name to "green",
            TradingOrderColor.SELL.name to "red",
            TradingOrderColor.BOTH.name to "orange"
        )
    ) +
    labs(
        title = "${output.tradingAlgorithmType} — ${output.securityIdentifier.tickerSymbol}",
        x = "Trading day",
        y = "Stock Price",
        color = "Trading Order"
    )
}

gggrid(
    plots,
    ncol = 1,
    hspace = 200,
    vspace = 200,
    align = true,
    fit = true
) + ggsize(4000, 4000)

<path d="M178.27291308494466 33.32772046563241 L178.27291308494466 33.32772046563241 L211.90931178021725 34.767706677150045 L245.5457104754898 39.03956055022243 L279.1821091707624 37.887583200515905 L312.81850786603496 35.10368473951968 L346.45490656130755 32.70375290729763 L380.0913052565801 28.81588502884543 L413.7277039518527 26.223960199313296 L447.36410264712526 28.2878946033197 L481.00050134239785 26.079975134253743 L514.6369000376704 23.248049375160974 L548.273298732943 22.864082746386828 L581.9096974282156 19.744167491328938 L615.5460961234882 21.7121053966803 L649.1824948187607 16.67227953436759 L682.8188935140333 18.160215580597736 L716.4552922093059 19.072192000743655 L750.0916909045784 16.67227953436759 L783.728089599851 17.824218152382088 L817.3644882951236 18.928187569838087 L851.0008869903962 21.32811940206014 L884.6372856856688 19.648170992673897 L918.2736843809413 21.376127334310667 L951.9100830762139 18.83221043702906 L985.5464817714865 23.44006173831707 L1019.182880466759 24.20803372755742 L1052.8192791620318 23.44006173831707 L1086.4556778573042 23.824028367091245 L1120.0920765525766 25.072002215452784 L1153.7284752478495 33.85571089115814 L1187.364873943122 34.38370131668387 L1221.0012726383943 33.663737259694074 L1254.6376713336672 34.9116917422096 L1288.2740700289396 35.535678666390396 L1321.910468724212 36.59162078574985 L1355.546867419485 36.9276182139655 L1389.1832661147573 35.87165672876003 L1422.8196648100297 36.25564272338019 L1456.4560635053026 38.46358155829216 L1490.092462200575 35.39165486963881 L1523.728860895848 32.511759909987546 L1557.3652595911203 32.511759909987546 L1591.0016582863927 30.30384044092159 L1624.6380569816656 30.73583436779228 L1658.274455676938 31.023804497911385 L1691.9108543722105 30.159836010016022 L1725.5472530674833 26.943943622149106 L1759.1836517627557 28.67188059793986 L1792.8200504580282 21.80810189533534 L1826.456449153301 26.127963700658256 L1860.0928478485735 43.455399488354345 L1893.7292465438459 43.359402989699305 L1927.3656452391187 47.72727272727272 L1961.0020439343912 46.14332081654155 L1994.6384426296636 47.4392832313076 L2028.2748413249365 46.767307740722316 L2061.911240020209 45.087359331336074 L2095.5476387154813 43.5514153528554 L2129.184037410754 41.2954685856929 L2162.8204361060266 40.62349309510762 L2196.456834801299 42.92742842867463 L2230.093233496572 43.16742935823524 L2263.7296321918443 41.2954685856929 L2297.366030887117 40.431519463643525 L2331.0024295823896 41.103494954228836 L2364.638828277662 39.03956055022243 L2398.275226972935 40.76749752601319 L2431.9116256682073 42.35144943674436 L2465.5480243634797 40.383511531393 L2499.1844230587526 43.359402989699305 L2532.820821754025 30.159836010016022 L2566.4572204492974 33.7597143925031 L2600.0936191445703 29.343856088525172 L2633.7300178398427 27.471934047674836 L2667.366416535115 26.511949695278417 L2701.002815230388 22.3360923208611 L2734.6392139256604 22.960059879195853 L2768.275612620933 23.10406431010142 L2801.9120113162057 27.27992168451874 L2835.548410011478 32.89574590460771 L2869.184808706751 31.599802855687642 L2902.8212074020234 33.471744262383964 L2936.457606097296 31.887772985806777 L2970.0940047925687 27.1839251858637 L3003.730403487841 25.743977706038066 L3037.3668021831136 26.319956697968337 L3071.0032008783865 25.119990781857297 L3104.639599573659 24.784012719487663 L3138.2759982689313 21.568120331620747 L3171.912396964204 21.13612640475006 L3205.5487956594766 22.04810282489595 L3239.185194354749 12.448394861853728 L3272.821593050022 4.576662606294292 L3306.4579917452943 2.944702763312563 L3340.0943904405667 2.2727272727272805 L3373.7307891358396 6.496573213549112 L3407.367187831112 6.8805785740152885 L3441.0035865263844 5.344634595534615 L3474.6399852216573 7.64855056325564 L3508.2763839169297 9.856470032321596 L3541.912782612202 11.728411439017918 L3575.549181307475 12.736403723664864 L3609.1855800027474 11.392433376648285 L3642.82197869802 13.21636685109408 L3676.4583773932927

In [ ]:
%use lets-plot

import application.tester.TradingAlgorithmBackTester
import data.repository.historical_data.HistoricalMarketDataProvider
import domain.algorithm.TradingAlgorithm
import domain.market.security.SecurityIdentifier
import domain.tax.Taxation
import domain.trader.TradingOrder
import kotlinx.coroutines.async
import kotlinx.coroutines.awaitAll
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.runBlocking
import kotlin.time.Instant

enum class TradingOrderColor {
    BUY,
    SELL,
    BOTH,
    NOTHING,
}

//===========================================================//
//===========================================================//

// Config
val algorithm = TradingAlgorithm.Type.TACPP46
val taxation = Taxation.Type.Hungary
val startCapital = 1000.0
val startDate = Instant.parse("2025-06-01T00:00:00Z")
val endDate = Instant.parse("2026-01-01T00:00:00Z")

val listOfOutput = runBlocking {
    coroutineScope {
        HistoricalMarketDataProvider.getAllSecurityIdentifiers().getOrThrow().map {
            async {
                TradingAlgorithmBackTester(
                    type = algorithm,
                    securityIdentifier = it,
                    startingCapital = startCapital,
                    taxation = taxation,
                    from = startDate,
                    to = endDate
                ).runBackTest()
            }
        }.awaitAll()
    }
}

val plots = listOfOutput.map { output ->
    output.tradingOrders.map { order ->
        if(order.buy == null && order.sell == null) TradingOrderColor.NOTHING
        else if(order.buy != null && order.sell != null) TradingOrderColor.BOTH
        else if(order.buy != null && order.sell == null) TradingOrderColor.BUY
        else TradingOrderColor.SELL
    }
}.first()
plots